# 05 Cross-Currency Basis

**Book:** *Fixed Income Relative Value Analysis (2nd ed.)*  
**Focus:** Chapter 15 cross-currency basis swaps, CIP-style decomposition, and FX-hedged bond intuition.


## Goals

1. Load available FX spot and USD funding proxies.
2. Build a notebook structure for cross-currency basis analysis.
3. Document the missing curves and basis quotes needed for a proper implementation.


In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

REPO_ROOT = Path("/Users/zelin/Desktop/PA Investment/Invest_strategy")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from alpha_research.quant_data.api import get_data

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

START = "2024-02-26"
END = "2026-02-27"
BOOK_PDF = Path(r"""/Users/zelin/Desktop/阅读学习/Fixed Income Relative Value Analysis + Website A Practitioner’s Guide to the Theory, Tools, and Trades 2nd.pdf""")

print("Expected environment: conda activate ibkr-analytics && export PYTHONPATH=.")
print("Book PDF exists:", BOOK_PDF.exists())
print("Repo root:", REPO_ROOT)


In [ ]:
FRED_DIR = REPO_ROOT / "data" / "market_data" / "fred"
PRICES_DIR = REPO_ROOT / "data" / "market_data" / "prices"

def _filter_date(frame: pd.DataFrame, start=START, end=END, date_col="date"):
    out = frame.copy()
    out[date_col] = pd.to_datetime(out[date_col])
    return out[(out[date_col] >= start) & (out[date_col] <= end)]

def load_fred_series(series_ids, start=START, end=END):
    frames = []
    for parquet_file in sorted(FRED_DIR.glob("*.parquet")):
        df = pd.read_parquet(parquet_file)
        if "series_id" not in df.columns:
            continue
        sub = df[df["series_id"].isin(series_ids)]
        if not sub.empty:
            frames.append(_filter_date(sub, start=start, end=end))

    if not frames:
        return pd.DataFrame()

    joined = pd.concat(frames, ignore_index=True).drop_duplicates(["date", "series_id"])
    wide = (
        joined.pivot(index="date", columns="series_id", values="value")
        .sort_index()
        .apply(pd.to_numeric, errors="coerce")
    )
    wide.index = pd.to_datetime(wide.index)
    return wide

def load_local_price_series(tickers, start=START, end=END, value_col="close"):
    frames = []
    for parquet_file in sorted(PRICES_DIR.glob("*.parquet")):
        df = pd.read_parquet(parquet_file)
        if "ticker" not in df.columns or value_col not in df.columns:
            continue
        sub = df[df["ticker"].isin(tickers)]
        if not sub.empty:
            frames.append(_filter_date(sub, start=start, end=end))

    if not frames:
        return pd.DataFrame()

    joined = pd.concat(frames, ignore_index=True).drop_duplicates(["date", "ticker"])
    wide = (
        joined.pivot(index="date", columns="ticker", values=value_col)
        .sort_index()
        .apply(pd.to_numeric, errors="coerce")
    )
    wide.index = pd.to_datetime(wide.index)
    return wide


In [ ]:
fx = load_local_price_series(["EURUSD=X", "GBPUSD=X", "USDJPY=X", "USDCAD=X"])
usd_rates = load_fred_series(["SOFR", "DFEDTARU", "DGS2", "DGS5", "DGS10"]).dropna()

fx.tail(), usd_rates.tail()


In [ ]:
fx.plot(title="FX Spot Proxies")
plt.show()


## Basis decomposition scaffold

A practical CCBS implementation needs:

- domestic and foreign OIS/reference-rate curves
- FX forwards or forward points
- day-count and reset conventions
- actual cross-currency basis quotes

Those are not currently present in the local lake, so use this notebook as a structured placeholder.


In [ ]:
required_inputs = pd.DataFrame(
    {
        "category": [
            "FX spot",
            "FX forwards",
            "USD reference curve",
            "foreign reference curve",
            "CCBS quoted basis",
        ],
        "available_locally": [True, False, True, False, False],
        "notes": [
            "EURUSD / GBPUSD / USDJPY / USDCAD in local yfinance cache",
            "not in current lake",
            "SOFR + Treasury proxies available",
            "not in current lake",
            "not in current lake",
        ],
    }
)
required_inputs


In [ ]:
# Placeholder covered-interest-parity style decomposition.
# TODO:
# 1. ingest FX forward points
# 2. ingest EUR / GBP / JPY OIS curves
# 3. compare implied basis to quoted basis
# 4. analyze issuance / investment examples from the book


## Validation path

Once data is available, extend this notebook to:

- reconstruct synthetic funding in foreign currency
- compare bond yields after FX hedging
- isolate residual basis dislocations across tenor and currency
